In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/annotated_output.vcfgz
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/filtered_output.vcfgz
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/strains_metadata.csv
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/vcf_files/annotated_output.vcf
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/vcf_files/filtered_output.vcf
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/phenotype_matrix/SRA_to_genome_name.csv
/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhe

In [2]:
!pip install cyvcf2 --no-deps -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 55.3 MB/s eta 0:00:00


In [3]:
import shutil
import os

src = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/filtered_output.vcfgz"
dst = "/kaggle/working/filtered_output.vcf.gz"
shutil.copy(src, dst)
print(f"Copied: {os.path.getsize(dst) / 1e9:.2f} GB")

Copied: 1.10 GB


In [4]:
"""
VCF → INTEGER SNP MATRIX (Arcadia convention)
Missing = ref allele (matches Arcadia BSLMM paper)
"""

import os
import numpy as np
import pandas as pd
from cyvcf2 import VCF
from datetime import datetime
import pickle
from pathlib import Path

VCF_PATH = "/kaggle/working/filtered_output.vcf.gz"
SRA_MAP_PATH = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/phenotype_matrix/SRA_to_genome_name.csv"
PHENO_PATH = "/kaggle/input/datasets/faiazabrarzaman/arcadia-presenceabsence-v2-ecoli-rawdownload/Ecoli_AMR_GenotypePhenotype/Ecoli_AMR_GenotypePhenotype/phenotype_matrix/phenotype_matrix_08302024.csv"
OUTPUT_DIR = Path("/kaggle/working")

# CONFIG (Arcadia convention)
MIN_MAF = 0.01           # 1% minor allele frequency
SNP_ONLY = True
PROGRESS_INTERVAL = 100000

print("="*70)
print("VCF → SNP MATRIX (Arcadia convention: missing=ref)")
print("="*70)

# Load metadata
sra_map = pd.read_csv(SRA_MAP_PATH)
pheno = pd.read_csv(PHENO_PATH)

# Sample selection
vcf = VCF(VCF_PATH)
vcf_samples = np.array(vcf.samples, dtype=str)
sra_to_genome = dict(zip(sra_map["SRA.Accession"], sra_map["Genome.Name"]))
pheno_genomes = set(pheno["Genome.Name"])

selected_indices = []
selected_genomes = []
selected_sra = []

for i, sra_id in enumerate(vcf_samples):
    if sra_id in sra_to_genome:
        gname = sra_to_genome[sra_id]
        if gname in pheno_genomes:
            selected_indices.append(i)
            selected_genomes.append(gname)
            selected_sra.append(sra_id)

selected_indices = np.array(selected_indices)
n_sel = len(selected_indices)
print(f"Selected samples: {n_sel}")

MIN_MAF_ABS = max(1.0 / n_sel, MIN_MAF)
print(f"MIN_MAF: {MIN_MAF_ABS:.4f}")

# Stream
print("\nStreaming variants (Arcadia convention)...")

snp_columns = []
snp_positions = []
n_streamed = 0
n_indel = 0
n_low_maf = 0
t_start = datetime.now()

for v in vcf:
    n_streamed += 1
    
    if n_streamed % PROGRESS_INTERVAL == 0:
        elapsed = (datetime.now() - t_start).total_seconds()
        rate = n_streamed / elapsed
        print(f"  Streamed {n_streamed:,} | Kept {len(snp_columns):,} | "
              f"Time: {elapsed:.0f}s ({rate:.0f} var/s)")
    
    # SNP-only
    ref = (v.REF or "").upper()
    alts = [(a or "").upper() for a in (v.ALT or [])]
    if SNP_ONLY:
        if len(ref) != 1 or not alts or any(len(a) != 1 for a in alts):
            n_indel += 1
            continue
    
    # Genotypes — Arcadia convention: missing → 0 (ref)
    genos = v.genotypes
    g_sel = [genos[i] for i in selected_indices]
    
    col = np.zeros(n_sel, dtype=np.int8)  # default 0 = ref
    for i, g in enumerate(g_sel):
        if g is None:
            continue  # stays 0
        a = g[0]
        if a is None or a < 0:
            continue  # missing → ref (stays 0)
        col[i] = a  # 0=ref, 1=alt1, 2=alt2, ...
    
    # MAF filter (now on assumption-corrected genotypes)
    n_ref = (col == 0).sum()
    n_alt = (col > 0).sum()
    minor = min(n_ref, n_alt)
    maf = minor / n_sel
    
    if maf < MIN_MAF_ABS:
        n_low_maf += 1
        continue
    
    snp_columns.append(col)
    snp_positions.append(f"{v.CHROM}:{v.POS}:{ref}>{','.join(alts)}")

vcf.close()
elapsed = (datetime.now() - t_start).total_seconds()

print(f"\n{'='*70}")
print(f"COMPLETE — {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"{'='*70}")
print(f"  Total streamed: {n_streamed:,}")
print(f"  Skipped (indel): {n_indel:,}")
print(f"  Skipped (low MAF): {n_low_maf:,}")
print(f"  KEPT: {len(snp_columns):,}")

# Build matrix
X_snp = np.array(snp_columns, dtype=np.int8).T
print(f"\n  Shape: {X_snp.shape}")
print(f"  Memory: {X_snp.nbytes / 1e9:.2f} GB")
print(f"  Value distribution: {dict(zip(*np.unique(X_snp, return_counts=True)))}")

# Save
output = {
    "X_snp": X_snp,
    "snp_positions": np.array(snp_positions),
    "selected_genome_names": np.array(selected_genomes),
    "selected_sra_ids": np.array(selected_sra),
    "n_streamed": n_streamed,
    "n_kept": len(snp_columns),
    "config": {
        "MIN_MAF": MIN_MAF_ABS,
        "SNP_ONLY": SNP_ONLY,
        "missing_handling": "Arcadia convention: missing → ref (0)",
    },
}

OUTPUT_PATH = OUTPUT_DIR / "arcadia_v2_snp_matrix.pkl"
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(output, f, protocol=4)

print(f"\n✅ Saved: {OUTPUT_PATH}")
print(f"   File size: {OUTPUT_PATH.stat().st_size / 1e9:.2f} GB")

VCF → SNP MATRIX (Arcadia convention: missing=ref)
Selected samples: 6962
MIN_MAF: 0.0100

Streaming variants (Arcadia convention)...
  Streamed 100,000 | Kept 25,915 | Time: 371s (270 var/s)
  Streamed 200,000 | Kept 49,767 | Time: 745s (268 var/s)
  Streamed 300,000 | Kept 76,168 | Time: 1121s (268 var/s)
  Streamed 400,000 | Kept 101,140 | Time: 1495s (268 var/s)
  Streamed 500,000 | Kept 125,895 | Time: 1869s (268 var/s)
  Streamed 600,000 | Kept 154,519 | Time: 2243s (268 var/s)
  Streamed 700,000 | Kept 180,761 | Time: 2608s (268 var/s)
  Streamed 800,000 | Kept 205,588 | Time: 2969s (269 var/s)
  Streamed 900,000 | Kept 231,588 | Time: 3339s (270 var/s)
  Streamed 1,000,000 | Kept 256,611 | Time: 3712s (269 var/s)
  Streamed 1,100,000 | Kept 280,043 | Time: 4086s (269 var/s)
  Streamed 1,200,000 | Kept 302,105 | Time: 4466s (269 var/s)
  Streamed 1,300,000 | Kept 326,538 | Time: 4851s (268 var/s)
  Streamed 1,400,000 | Kept 350,728 | Time: 5236s (267 var/s)
  Streamed 1,500,000 